# GraphPack experiments — Gemma 4 Developer Agent Paper Track

Reproduces every number in the paper: localization on the competition's public dev tasks
with Gemma 4 E4B / E2B under fixed token budgets.

**Before running:** Settings → Accelerator **GPU (L4x4 or T4x2)**, Internet **on** (only for `pip install`),
Add Input → the competition dataset, the Gemma 4 model(s), and the `graphpack` code
(upload the `gemma4-paper-track` folder as a Kaggle Dataset, or `git clone` it).
Results land in `/kaggle/working/results/` and `/kaggle/working/report/`.

In [ ]:
import os, sys, glob, subprocess, time, json
# where the graphpack code is (Kaggle Dataset upload or a git clone)
CODE_DIR = next((p for p in glob.glob('/kaggle/input/*/gemma4-paper-track') + glob.glob('/kaggle/input/graphpack*')
                 + ['/kaggle/working/gemma4-paper-track'] if os.path.isdir(os.path.join(p, 'graphpack'))), None)
assert CODE_DIR, 'add the graphpack code as an input dataset (folder containing graphpack/)'
sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR   # for the `!python -m graphpack...` subprocesses
os.chdir('/kaggle/working')
os.makedirs('results', exist_ok=True)
print(CODE_DIR)
!python {CODE_DIR}/tests/test_smoke.py

## 1. Look at the competition data, then build `tasks.jsonl`
Each task needs: an id, the issue text, the gold patch, and a directory with the **pre-patch** repository.
Run the inspect cell, read the printed keys, and set `SRC` / `REPO_TEMPLATE` / field names below.

In [ ]:
COMP = sorted(glob.glob('/kaggle/input/*gemma-4-developer-agent*'))
print(COMP)
!python -m graphpack.prepare_tasks --inspect {COMP[0] if COMP else '/kaggle/input'}

In [ ]:
# EDIT these three lines after looking at the inspect output
SRC = f'{COMP[0]}/dev/tasks.jsonl'                       # the task table
REPO_TEMPLATE = COMP[0] + '/dev/repos/{instance_id}'     # pre-patch snapshot per task
EXTRA = ''   # e.g. '--field-problem-statement issue --field-patch gold_patch'
!python -m graphpack.prepare_tasks --src {SRC} --repo-template "{REPO_TEMPLATE}" {EXTRA} --out tasks.jsonl
!wc -l tasks.jsonl

## 2. Retrieval-only baselines (CPU, minutes)

In [ ]:
for r in ['bm25', 'ppr', 'graphpack']:
    !python -m graphpack.run --tasks tasks.jsonl --out results/retrieval.jsonl --method retrieval --ranker {r}
!python -m graphpack.report results/*.jsonl --out-dir report

## 3. Serve Gemma 4 with vLLM
Point `MODEL_PATH` at the model you added as input (the folder with `config.json`).
Use the same checkpoint family the main competition allows (E4B, and E2B for the scale comparison).

In [ ]:
MODEL_PATH = sorted(glob.glob('/kaggle/input/gemma-4*/**/config.json', recursive=True))
print(MODEL_PATH)
MODEL_PATH = os.path.dirname(MODEL_PATH[0])   # EDIT if the wrong one is picked
try:
    import vllm  # noqa
except ImportError:
    !pip install -q vllm
def serve(model_path, name, port=8000, tp=None):
    import torch
    tp = tp or torch.cuda.device_count()
    log = open(f'vllm_{name}.log', 'w')
    proc = subprocess.Popen(['vllm', 'serve', model_path, '--served-model-name', name, '--port', str(port),
                             '--max-model-len', '32768', '--tensor-parallel-size', str(tp),
                             '--gpu-memory-utilization', '0.90'], stdout=log, stderr=subprocess.STDOUT)
    import urllib.request
    for _ in range(180):
        try:
            urllib.request.urlopen(f'http://localhost:{port}/v1/models', timeout=2); break
        except Exception:
            time.sleep(5)
    else:
        raise RuntimeError(open(f'vllm_{name}.log').read()[-3000:])
    os.environ.update(LLM_BASE_URL=f'http://localhost:{port}/v1', LLM_API_KEY='EMPTY', LLM_MODEL=name)
    return proc
server = serve(MODEL_PATH, 'gemma4-e4b')

## 4. One-shot localization: budget × ranker × packing (main experiment)

In [ ]:
MODEL = 'gemma4-e4b'
GRID = [('bm25', 'bodies'), ('graphpack', 'bodies'), ('graphpack', 'skeleton'), ('graphpack', 'hybrid')]
for budget in [2000, 4000, 8000, 16000]:
    for ranker, mode in GRID:
        !python -m graphpack.run --tasks tasks.jsonl --out results/oneshot_{MODEL}.jsonl --method oneshot \
            --ranker {ranker} --pack {mode} --budget {budget} --model {MODEL}

## 5. Agents: grep tools vs graph tools vs graph tools + 2k starting pack

In [ ]:
for tools, sp in [('grep', 0), ('graph', 0), ('graph', 2000)]:
    !python -m graphpack.run --tasks tasks.jsonl --out results/agent_{MODEL}.jsonl --method agent \
        --tools {tools} --turns 15 --start-pack {sp} --context-limit 24000 --model {MODEL}

## 6. Scale: repeat the key configs with E2B
Stop the E4B server first, add the E2B model as input, then run.

In [ ]:
server.terminate(); server.wait()
E2B_PATH = [p for p in glob.glob('/kaggle/input/gemma-4*/**/config.json', recursive=True) if 'e2b' in p.lower()]
print(E2B_PATH)
E2B_PATH = os.path.dirname(E2B_PATH[0])   # EDIT if the wrong one is picked
server = serve(E2B_PATH, 'gemma4-e2b')
MODEL = 'gemma4-e2b'
for budget in [2000, 4000, 8000, 16000]:
    for ranker, mode in [('bm25', 'bodies'), ('graphpack', 'hybrid')]:
        !python -m graphpack.run --tasks tasks.jsonl --out results/oneshot_{MODEL}.jsonl --method oneshot \
            --ranker {ranker} --pack {mode} --budget {budget} --model {MODEL}
for tools, sp in [('grep', 0), ('graph', 2000)]:
    !python -m graphpack.run --tasks tasks.jsonl --out results/agent_{MODEL}.jsonl --method agent \
        --tools {tools} --turns 15 --start-pack {sp} --model {MODEL}

## 7. Tables and figure for the paper
`report/results.md` has every table; `report/budget_curve.png` is Figure 1.
Config names do not contain the model, so report each model's files separately.

In [ ]:
for m in ['gemma4-e4b', 'gemma4-e2b']:
    files = ' '.join(glob.glob(f'results/*{m}.jsonl') + ['results/retrieval.jsonl'])
    !python -m graphpack.report {files} --out-dir report/{m}
!python -m graphpack.report results/retrieval.jsonl --out-dir report/retrieval
!cd /kaggle/working && zip -qr graphpack_results.zip results report && ls -la